In [1]:
VERSION = 2

import pandas as pd
import numpy as np
from catboost import CatBoostRegressor, Pool

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shap
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv(r"C:\Users\Pratham Shah\Downloads\Tubingen\Tabular_Models_Dataset\Updated02\community_index_v9.csv")
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

Loaded: 48,000 rows × 168 columns


In [3]:
# feature selection controlled by version

r0_cols     = [f"R0_c{k}"     for k in range(1, 21)]
active_cols = [f"active_c{k}" for k in range(1, 21)]

if VERSION == 1:
    species_cols = [f"Species_{i:04d}" for i in range(2, 102)]
    feature_cols = species_cols + r0_cols + active_cols + ['community_size', 'diversity_level']
    version_label = "Full (species + nutrients + size)"

elif VERSION == 2:
    feature_cols = r0_cols + active_cols + ['community_size', 'diversity_level']
    version_label = "Nutrients + size (no species)"

elif VERSION == 3:
    feature_cols = r0_cols + active_cols
    version_label = "Pure nutrients only (no species, no size)"

else:
    raise ValueError("VERSION must be 1, 2, or 3")

print(f"\nVersion: {version_label}")
print(f"Features: {len(feature_cols)}")

missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise KeyError(f"{len(missing)} columns not found. First 3: {missing[:3]}\n"
                   f"Make sure you are loading community_index_v8_backup.csv (30,000 rows), "
                   f"not community_dataset_v8_backup.csv (792,000 rows).")

X = df[feature_cols]
y = df['community_suppression']
print(f"X: {X.shape}   y range: [{y.min():.3f}, {y.max():.3f}]")


Version: Nutrients + size (no species)
Features: 42
X: (48000, 42)   y range: [0.000, 0.999]


In [4]:
# train test split

strat_key = df['community_size'].astype(str) + "_" + df['diversity_level'].astype(str)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=strat_key
)
print(f"\nTrain: {X_train.shape[0]:,}   Test: {X_test.shape[0]:,}")

# Wrap train and test in CatBoost Pool objects.
# Pool(data, label): packages features and target together.
# CatBoost uses Pool internally for all operations; passing Pools directly
# avoids repeated data conversion overhead during training.
# cat_features: list of column indices that are categorical.
#   All our features are numeric so this is an empty list.

train_pool = Pool(X_train, label=y_train, cat_features=[])
test_pool  = Pool(X_test,  label=y_test,  cat_features=[])


Train: 38,400   Test: 9,600


In [5]:
# ridge regression baseline

scaler = StandardScaler()
ridge  = Ridge(alpha=1.0).fit(scaler.fit_transform(X_train), y_train)
yp_ridge  = ridge.predict(scaler.transform(X_test))
r2_ridge  = r2_score(y_test, yp_ridge)
mae_ridge = mean_absolute_error(y_test, yp_ridge)
print(f"\nRidge R²={r2_ridge:.4f}  MAE={mae_ridge:.4f}")


Ridge R²=0.8392  MAE=0.1015


In [6]:
# catboost initial fir

# note about l2_leaf_reg:
#  L2 regularisation on leaf values. Equivalent to
#  reg_lambda in XGBoost. Default is 3.0 (stronger than
#  XGBoost's default of 1.0)

model_init = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=3.0,
    subsample=0.8,
    colsample_bylevel=0.8,
    early_stopping_rounds=30,
    use_best_model=True,
    eval_metric='RMSE',
    random_seed=42,
    thread_count=-1,
    verbose=0
)

# Pass Pool objects directly 
# eval_set here is the test Pool for early stopping monitoring.
model_init.fit(train_pool, eval_set=test_pool)

yp_init  = model_init.predict(X_test)
r2_init  = r2_score(y_test, yp_init)
mae_init = mean_absolute_error(y_test, yp_init)
print(f"CatBoost initial:  R²={r2_init:.4f}  MAE={mae_init:.4f}  "
      f"best_iter={model_init.best_iteration_}")

CatBoost initial:  R²=0.8625  MAE=0.0789  best_iter=352


In [7]:
# hyperparameter tuning

param_grid = {
    'depth':            [4,5, 6,7, 8],
    # depth is CatBoost's primary complexity control.
    # depth=4 → 16 leaves, depth=8 → 256 leaves.

    'learning_rate':    [0.01, 0.05,0.08, 0.1],

    'l2_leaf_reg':      [1.0,2.0, 3.0,5.0,8.0, 10.0],
    # CatBoost-specific. Higher = more regularisation.
    # Default 3.0 is already stronger than XGBoost/LightGBM defaults.
    # Try 10.0 for VERSION 3 (pure nutrient) where overfitting is more likely.

    'colsample_bylevel':[0.6, 0.8, 1.0],
}


cv = KFold(n_splits=3, shuffle=True, random_state=42)

base_est = CatBoostRegressor(
    iterations=300,
    subsample=0.8,
    early_stopping_rounds=None,   # disabled inside GridSearchCV
    use_best_model=False,         # disabled inside GridSearchCV
    eval_metric='RMSE',
    random_seed=42,
    thread_count=-1,
    verbose=0
)

grid_search = GridSearchCV(
    estimator=base_est,
    param_grid=param_grid,
    cv=cv,
    scoring='neg_mean_squared_error',
    n_jobs=1,
    # n_jobs=1 here (not -1): CatBoost already uses all threads internally
    # via thread_count=-1. Running outer parallelism on top causes resource
    # contention and can crash. Keep outer n_jobs=1 for CatBoost.
    verbose=1,
    refit=True
)
grid_search.fit(X_train, y_train)

print(f"\nBest params: {grid_search.best_params_}")
print(f"Best CV RMSE: {np.sqrt(-grid_search.best_score_):.4f}")

Fitting 3 folds for each of 360 candidates, totalling 1080 fits

Best params: {'colsample_bylevel': 1.0, 'depth': 6, 'l2_leaf_reg': 8.0, 'learning_rate': 0.05}
Best CV RMSE: 0.1403


In [8]:
# final model

model_final = CatBoostRegressor(
    **grid_search.best_params_,
    iterations=1000,
    subsample=0.8,
    early_stopping_rounds=40,
    use_best_model=True,
    eval_metric='RMSE',
    random_seed=42,
    thread_count=-1,
    verbose=50
)

model_final.fit(train_pool, eval_set=test_pool)

yp_final   = model_final.predict(X_test)
r2_final   = r2_score(y_test, yp_final)
mae_final  = mean_absolute_error(y_test, yp_final)
rmse_final = np.sqrt(mean_squared_error(y_test, yp_final))

print(f"\nCatBoost final:  R²={r2_final:.4f}  MAE={mae_final:.4f}  RMSE={rmse_final:.4f}")
print(f"Trees used: {model_final.best_iteration_}")

0:	learn: 0.3617427	test: 0.3639456	best: 0.3639456 (0)	total: 9.15ms	remaining: 9.14s
50:	learn: 0.1538767	test: 0.1540632	best: 0.1540632 (50)	total: 419ms	remaining: 7.79s
100:	learn: 0.1422259	test: 0.1427388	best: 0.1427388 (100)	total: 828ms	remaining: 7.37s
150:	learn: 0.1404322	test: 0.1411000	best: 0.1411000 (150)	total: 1.28s	remaining: 7.2s
200:	learn: 0.1399409	test: 0.1406447	best: 0.1406447 (200)	total: 1.71s	remaining: 6.81s
250:	learn: 0.1397237	test: 0.1404700	best: 0.1404700 (250)	total: 2.12s	remaining: 6.32s
300:	learn: 0.1396341	test: 0.1404092	best: 0.1404092 (300)	total: 2.47s	remaining: 5.73s
350:	learn: 0.1395985	test: 0.1403931	best: 0.1403930 (349)	total: 2.86s	remaining: 5.28s
Stopped by overfitting detector  (40 iterations wait)

bestTest = 0.1403930431
bestIteration = 349

Shrink model to first 350 iterations.

CatBoost final:  R²=0.8624  MAE=0.0789  RMSE=0.1404
Trees used: 349


In [9]:
# per condition breakdown

results_df = pd.DataFrame({
    'y_true':          y_test.values,
    'y_pred':          yp_final,
    'diversity_level': df.loc[X_test.index, 'diversity_level'].values,
    'community_size':  df.loc[X_test.index, 'community_size'].values,
})

print("\nBy diversity level:")
for d, grp in results_df.groupby('diversity_level'):
    print(f"  D={d:<4} R²={r2_score(grp.y_true, grp.y_pred):.4f}  "
          f"MAE={mean_absolute_error(grp.y_true, grp.y_pred):.4f}  n={len(grp):,}")

print("\nBy community size:")
for n, grp in results_df.groupby('community_size'):
    print(f"  N={n:<4} R²={r2_score(grp.y_true, grp.y_pred):.4f}  "
          f"MAE={mean_absolute_error(grp.y_true, grp.y_pred):.4f}  n={len(grp):,}")


By diversity level:
  D=1    R²=0.8941  MAE=0.0720  n=3,000
  D=2    R²=0.8785  MAE=0.0758  n=2,400
  D=4    R²=0.3570  MAE=0.0798  n=1,800
  D=8    R²=0.6216  MAE=0.0828  n=1,200
  D=16   R²=0.4373  MAE=0.0969  n=1,200

By community size:
  N=12   R²=0.6922  MAE=0.1395  n=1,920
  N=16   R²=0.7800  MAE=0.1116  n=1,920
  N=24   R²=0.9088  MAE=0.0652  n=1,920
  N=32   R²=0.9260  MAE=0.0509  n=1,920
  N=48   R²=0.9788  MAE=0.0272  n=1,920


In [10]:
# shap analysis

print('shap analysis')

# CatBoost has its own built-in SHAP implementation accessible via
# get_feature_importance(type='ShapValues'), which is fast and exact.
# However, shap.TreeExplainer also supports CatBoost 
# we use it here to keep the interface identical across all three
# model scripts so results are directly comparable.
explainer   = shap.TreeExplainer(model_final)
rng         = np.random.default_rng(42)
idx         = rng.choice(len(X_test), size=2000, replace=False)
shap_values = explainer.shap_values(X_test.iloc[idx])
mean_shap   = pd.Series(np.abs(shap_values).mean(0), index=feature_cols).sort_values(ascending=False)

print(f"\nTop 15 features by mean |SHAP|:")
print(mean_shap.head(15).round(5).to_string())

shap analysis

Top 15 features by mean |SHAP|:
R0_c4              0.10925
community_size     0.06150
R0_c11             0.04946
R0_c1              0.04836
R0_c17             0.04006
R0_c3              0.01413
R0_c16             0.01256
R0_c2              0.01140
R0_c8              0.01066
R0_c15             0.00771
R0_c7              0.00648
diversity_level    0.00631
R0_c12             0.00597
active_c19         0.00597
R0_c19             0.00562


In [13]:
# figures

output_name = f"catboost_v{VERSION}_results_02.png"

fig = plt.figure(figsize=(18, 14), dpi=120)
fig.suptitle(
    f'CatBoost — {version_label} — MiCRM v8\n'
    f'R²={r2_final:.3f}  MAE={mae_final:.3f}  RMSE={rmse_final:.3f}',
    fontsize=12, fontweight='bold', y=0.99
)

ax1 = fig.add_subplot(2, 3, 1)
for d, col in zip([1,2,4,8,16], ['#e41a1c','#ff7f00','#4daf4a','#377eb8','#984ea3']):
    mask = results_df['diversity_level'].values == d
    ax1.scatter(y_test.values[mask], yp_final[mask], alpha=0.15, s=2, c=col, label=f'D={d}')
ax1.plot([0,1],[0,1],'r--',lw=1.5)
ax1.set_xlabel('True suppression'); ax1.set_ylabel('Predicted')
ax1.set_title(f'Predicted vs Actual  R²={r2_final:.3f}')
ax1.legend(fontsize=7, markerscale=3)

ax2 = fig.add_subplot(2, 3, 2)
resid = y_test.values - yp_final
for n, col in zip([12,16,24,32,48], ['#1b9e77','#d95f02','#7570b3','#e7298a','#66a61e']):
    mask = results_df['community_size'].values == n
    ax2.scatter(yp_final[mask], resid[mask], alpha=0.15, s=2, c=col, label=f'N={n}')
ax2.axhline(0, color='r', linestyle='--', lw=1.5)
p5, p95 = np.percentile(resid, [5, 95])
ax2.axhline(p5,  color='grey', linestyle=':', lw=1, label=f'5th: {p5:.3f}')
ax2.axhline(p95, color='grey', linestyle=':', lw=1, label=f'95th: {p95:.3f}')
ax2.set_xlabel('Predicted'); ax2.set_ylabel('Residual')
ax2.set_title('Residuals by Community Size')
ax2.legend(fontsize=7, markerscale=3)

ax3 = fig.add_subplot(2, 3, 3)
top_n    = min(25, len(feature_cols))
top_shap = mean_shap.head(top_n)
def feat_col(f):
    if 'Species' in f: return '#2196F3'
    if 'R0'      in f: return '#4CAF50'
    if 'active'  in f: return '#FF9800'
    return '#9C27B0'
cols = [feat_col(f) for f in top_shap.index]
ax3.barh(range(top_n), top_shap.values[::-1], color=cols[::-1])
ax3.set_yticks(range(top_n))
ax3.set_yticklabels(top_shap.index[::-1], fontsize=7)
ax3.set_xlabel('Mean |SHAP|'); ax3.set_title(f'Top {top_n} Features (SHAP)')
ax3.legend(handles=[
    Patch(fc='#2196F3', label='Species'),
    Patch(fc='#4CAF50', label='R0'),
    Patch(fc='#FF9800', label='Active'),
    Patch(fc='#9C27B0', label='Structural'),
], loc='lower right', fontsize=7)

ax4 = fig.add_subplot(2, 3, 4)
divs   = sorted(results_df['diversity_level'].unique())
r2_div = [r2_score(results_df[results_df['diversity_level']==d]['y_true'],
                   results_df[results_df['diversity_level']==d]['y_pred']) for d in divs]
ax4.bar([str(d) for d in divs], r2_div, color='steelblue', edgecolor='navy')
ax4.axhline(r2_final, color='red', linestyle='--', lw=1.5, label=f'Overall R²={r2_final:.3f}')
ax4.set_xlabel('Diversity level'); ax4.set_ylabel('R²')
ax4.set_title('R² by Diversity Level'); ax4.set_ylim(0, 1.05); ax4.legend(fontsize=8)

ax5 = fig.add_subplot(2, 3, 5)
sizes  = sorted(results_df['community_size'].unique())
r2_sz  = [r2_score(results_df[results_df['community_size']==n]['y_true'],
                   results_df[results_df['community_size']==n]['y_pred']) for n in sizes]
ax5.bar([str(n) for n in sizes], r2_sz, color='darkorange', edgecolor='saddlebrown')
ax5.axhline(r2_final, color='red', linestyle='--', lw=1.5, label=f'Overall R²={r2_final:.3f}')
ax5.set_xlabel('Community size N'); ax5.set_ylabel('R²')
ax5.set_title('R² by Community Size'); ax5.set_ylim(0, 1.05); ax5.legend(fontsize=8)

ax6 = fig.add_subplot(2, 3, 6)
groups, values, colours = [], [], []
sp_s = mean_shap[[f for f in feature_cols if 'Species' in f]].sum()
if sp_s > 0:
    groups.append('Species'); values.append(sp_s); colours.append('#2196F3')
r0_s = mean_shap[r0_cols].sum()
groups.append('R0'); values.append(r0_s); colours.append('#4CAF50')
ac_s = mean_shap[active_cols].sum()
groups.append('Active'); values.append(ac_s); colours.append('#FF9800')
st_s = mean_shap[[f for f in feature_cols if f in ['community_size','diversity_level']]].sum()
if st_s > 0:
    groups.append('Structural'); values.append(st_s); colours.append('#9C27B0')
ax6.pie(values, labels=groups, colors=colours,
        autopct='%.1f%%', startangle=90, textprops={'fontsize': 10})
ax6.set_title('SHAP by Feature Group')

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(output_name, dpi=150, bbox_inches='tight')
plt.close()
print(f"\nPlot saved: {output_name}")


Plot saved: catboost_v2_results_02.png


In [ ]:
# save outputs 

model_save = f"catboost_v{VERSION}_model.cbm"
pred_save  = f"catboost_v{VERSION}_predictions.csv"

# CatBoost saves in its own .cbm (CatBoost Model) binary format.
# Reload with: m = CatBoostRegressor(); m.load_model('catboost_v1_model.cbm')
model_final.save_model(model_save)

pd.DataFrame({
    'y_true':          y_test.values,
    'y_pred':          yp_final,
    'residual':        y_test.values - yp_final,
    'diversity_level': results_df['diversity_level'].values,
    'community_size':  results_df['community_size'].values,
}).to_csv(pred_save, index=False)

print(f"Model saved:       {model_save}")
print(f"Predictions saved: {pred_save}")

In [12]:
# summary

print(f"\n{'='*60}")
print(f"SUMMARY — CatBoost VERSION {VERSION}: {version_label}")
print(f"{'='*60}")
print(f"Features:  {len(feature_cols)}")
print(f"R²:        {r2_final:.4f}")
print(f"MAE:       {mae_final:.4f}")
print(f"RMSE:      {rmse_final:.4f}")
print(f"Trees:     {model_final.best_iteration_}")
print(f"\nTop 5 features:")
for feat, val in mean_shap.head(5).items():
    print(f"  {feat:<25} {val:.5f}")


SUMMARY — CatBoost VERSION 2: Nutrients + size (no species)
Features:  42
R²:        0.8624
MAE:       0.0789
RMSE:      0.1404
Trees:     349

Top 5 features:
  R0_c4                     0.10925
  community_size            0.06150
  R0_c11                    0.04946
  R0_c1                     0.04836
  R0_c17                    0.04006
